# Fine-tuning RF-DETR Small on MOT17 (Kaggle)

Settings: **Accelerator = GPU** (T4 or P100), **Internet = On**. Then *Save Version > Save & Run All*.

Output: `/kaggle/working/rfdetr-small-mot17/last_ema.pth` (download it to `weights/` in the repo).

In [ ]:
EPOCHS = 20
BATCH_SIZE = 8
GRAD_ACCUM = 2
RESOLUTION = 512
DATA = "/tmp/data"
OUT = "/kaggle/working/rfdetr-small-mot17"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

## Code and dependencies
Torch is preinstalled with CUDA on Kaggle; only the extra packages are added.

In [ ]:
!git clone --depth 1 https://github.com/Louiscrrn/detr-tracking.git /kaggle/working/detr-tracking
%cd /kaggle/working/detr-tracking
!pip install -q 'rfdetr[train]>=1.12,<2' 'supervision>=0.31,<0.32' pycocotools
import os
import sys

sys.path.insert(0, "src")
os.environ["PYTHONPATH"] = "src"

## MOT17 (train sequences only, FRCNN folders: images are shared across detectors)

In [ ]:
!mkdir -p {DATA}
!wget -q -O {DATA}/MOT17.zip https://motchallenge.net/data/MOT17.zip
!cd {DATA} && unzip -q MOT17.zip 'MOT17/train/*-FRCNN/*' && rm MOT17.zip
!ls {DATA}/MOT17/train && df -h /tmp | tail -1

In [ ]:
!python scripts/prepare_mot17.py --root {DATA}/MOT17 --out {DATA}/annotations

## Training
Fixed number of epochs; the last EMA checkpoint is the one we evaluate.

In [ ]:
!python scripts/train_detector.py \
    --ann-dir {DATA}/annotations --image-root {DATA}/MOT17/train \
    --dataset-dir {DATA}/rfdetr --output-dir {OUT} \
    --epochs {EPOCHS} --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} \
    --resolution {RESOLUTION} --num-workers 4 --device cuda:0

## Keep only what we need in the output

In [ ]:
!rm -f {OUT}/checkpoint_best_*.pth
!ls -lh {OUT}
!cat {OUT}/metrics.csv | tail -5
!rm -rf /kaggle/working/detr-tracking